In [ ]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict,Annotated
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage,BaseMessage
from langgraph.graph.message import add_messages


In [ ]:
load_dotenv()

In [ ]:
llm=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [ ]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [ ]:
def chat_node(state: ChatState):

    # take user query from state
    messages = state['messages']

    # send to llm
    response = llm.invoke(messages)
    clean_response = AIMessage(
        content=response.content[0]['text']
    )
    # response store state
    return {'messages': [clean_response]}

In [ ]:
graph=StateGraph(ChatState)

graph.add_node("chat_node",chat_node)

graph.add_edge(START,"chat_node")
graph.add_edge("chat_node",END)

workflow=graph.compile()


In [ ]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of india')]
}

workflow.invoke(initial_state)['messages'][-1]

